# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

## 0. 제출 정보
- 이름: 권영서
- GitHub ID: kys02101-cmd
- 작성일: 2026-10-14
- 최종 제출 URL: https://github.com/kys02101-cmd/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape: customers (150, 6), products (100, 4), orders (300, 5), order_items (764, 5)
- 결측치: 4개 데이터셋 모두 전체 결측 셀 0개 (customers 0 / products 0 / orders 0 / order_items 0)
- 전체 중복: 4개 데이터셋 모두 전체 행 중복 0건
- 주요 ID 중복: `customer_id` 0, `product_id` 0, `order_id` 0, `order_item_id` 0. 단, `order_items.order_id`는 464건이 반복되는데 한 주문에 여러 상품이 담기기 때문에 자연스러운 반복임
- 타입 문제 후보: `orders.order_date`와 `customers.signup_date`가 날짜가 아닌 문자열(object)로 저장되어 있어 날짜형 변환이 필요함. 숫자 컬럼(`age`, `price`, `quantity`, `unit_price`)은 이미 int64라서 문자열 숫자 문제는 없음. `order_status`는 completed 184 / cancelled 64 / refunded 52

<!-- 캡처: shape, 결측 합계, 중복 수를 출력한 셀(STEP 3)과 dtypes 출력이 한 화면에 보이게 -->
![전처리 전 상태](images/step01_before.png)

### 결과 관찰
- 4개 파일 모두 결측치와 전체 중복이 0건이었다.
- 4개 파일 모두 고유 ID(PK)가 중복되지 않았다.
- `order_items.order_id` 반복 464건은 고유 주문 300건에 상세 764행이 붙은 구조에서 나온 값이다.
- 주문 상태는 completed, cancelled, refunded 3종뿐이었고 `complete`, `완료` 같은 다른 표기는 보이지 않았다.
- 날짜 2종(`order_date`, `signup_date`)이 문자열이다.

### 나의 해석과 판단
이번 샘플 데이터는 결측과 중복이 없어서 "지울 것"이 거의 없고, 먼저 처리할 것은 날짜 문자열의 날짜형 변환이다. 월별·요일별 분석(`order_month`, `order_dayofweek`)이 날짜형에 의존하기 때문이다. 그다음은 안 보이는 문제를 확인하는 일이다. 공백, 표기 차이, 0 이하 값, 변환 실패가 이번에는 없었는지 숫자로 확인해야 "없다"고 말할 수 있다. 문제가 없다는 결과도 확인한 기록이 있어야 근거가 된다. 코드 실행 성공과 데이터 정상은 같은 말이 아니다.

### 업무·분석적 의미
전처리 전 상태를 남겨 두면 나중에 행 수나 합계가 바뀌었을 때 "무엇이 원인인지" 비교할 기준점이 생긴다. 이번처럼 0건이라도 "원본은 이 상태였다"는 기록이 이후 Chapter 06 EDA 결과를 믿는 근거가 된다.

### 한계와 추가 확인 사항
이 데이터는 실습용 샘플이라 실제 업무 데이터처럼 지저분하지 않다. 실제 데이터라면 결측이 생기는 이유(입력 누락인지, 원래 없는 값인지)를 시스템 담당자에게 확인해야 한다. 또한 결측 0건은 "빈 칸이 없다"는 뜻일 뿐, 값이 정확하다는 뜻은 아니다.

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 문자열 컬럼의 앞뒤 공백 제거(`strip_string_columns`). NaN은 그대로 두고 값이 있는 칸만 `str.strip()` 적용. 실제로 공백이 있던 값은 0건이라 바뀐 값은 없음
- 숫자 변환 대상과 실패 건수: `products.price`, `order_items.quantity`, `order_items.unit_price`를 쉼표 제거 후 `pd.to_numeric(errors='coerce')`로 변환. 실패 건수는 price 0 / quantity 0 / unit_price 0
- 날짜 변환 대상과 실패 건수: `orders.order_date`(실패 0, 범위 2025-10-08 ~ 2026-10-07), `customers.signup_date`(실패 0, 범위 2023-10-13 ~ 2026-09-26)
- 결측 처리 기준: 나이 결측은 중앙값(40세)으로, 도시 결측은 `Unknown`으로 대체하는 규칙을 적용. 이번 데이터에서는 두 컬럼 모두 결측이 0건이라 실제로 채워진 값은 0건

<!-- 캡처: 문자열 정리, 숫자/날짜 변환 실패 건수 print, 결측 처리 후 missing_summary 출력 -->
![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
- 숫자 변환 실패 3개 컬럼 모두 0건, 날짜 변환 실패 2개 컬럼 모두 0건이었다.
- 나이는 19~69세, 중앙값 40세였다.
- 결측 처리 규칙을 적용한 뒤에도 값이 바뀐 행이 없었고 결측 합계는 계속 0이었다.
- `order_month`, `order_dayofweek` 파생 컬럼이 오류 없이 만들어졌다.

### 나의 해석과 판단
나이처럼 연속형 숫자는 평균보다 중앙값이 극단값의 영향을 덜 받아서 중앙값을 기준으로 삼았다. 도시처럼 범주형은 임의의 도시로 채우면 없는 정보를 만들어 내므로 `Unknown`이라는 별도 범주로 남겨 "모른다"는 사실이 분석에 드러나게 했다. `errors='coerce'`는 변환 못 한 값을 조용히 NaN/NaT로 바꾸기 때문에, 변환 후 실패 건수를 반드시 세어서 새로 생긴 결측을 놓치지 않으려 했다. 삭제는 그 행의 다른 정보까지 함께 버리고, 대체는 실제와 다른 값을 넣어 분포를 좁게 만든다. 둘 다 정보 손실이 생길 수 있다.

### 업무·분석적 의미
결측을 중앙값으로 채운 나이로 연령대별 구매 분석을 하면 실제보다 40세 근처에 사람이 몰려 보일 수 있다. `Unknown` 범주를 유지하면 "도시 정보가 없는 고객"을 따로 확인할 수 있다. 고유 ID, 핵심 날짜, 금액, 상태값은 근거 없이 대표값으로 대체하지 않는다는 원칙이 필요하다.

### 한계와 추가 확인 사항
이번에는 결측이 0건이라 중앙값 대체와 `Unknown` 처리의 효과는 실제로 검증하지 못했다. 규칙이 코드에 들어 있다는 것까지만 확인했다. 결측이 생기면 몇 건이 바뀌었는지와 바꾼 뒤 분포 변화를 함께 기록해야 한다.

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 전체 행 중복 0건이라 제거한 행 없음. `drop_duplicates()`는 규칙으로만 적용. 고유 ID 중복도 0건. `order_items.order_id` 반복 464건은 자연스러운 반복이라 유지(보류)
- 표준화한 범주값: `order_status`를 `status_map`(complete/완료 → completed, cancel/취소 → cancelled, refund/환불 → refunded)으로 표준화. 변경된 값은 0건, 표준화 후에도 completed 184 / cancelled 64 / refunded 52로 동일
- 허용값 밖 값: 허용값(completed, cancelled, refunded) 밖의 상태값 0건
- 이상값 후보: 나이 18세 미만/100세 초과 0건, `price <= 0` 0건, `quantity <= 0` 0건, `unit_price <= 0` 0건. 범위 확인 결과 quantity 1~5, price 5,000~200,000원, `line_total` 5,000~1,000,000원이고 IQR 기준(상한 1,020,375원) 초과는 0건

<!-- 캡처: 중복 확인, status 정리 전/후 value_counts, price/quantity/unit_price describe 와 0 이하 건수 -->
![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
- 전체 중복과 키 중복이 모두 0건이다.
- 주문 상태는 정리 전후가 같았고 허용값 밖 값도 없었다.
- 0 이하 가격·수량·단가와 비정상 나이는 발견되지 않았다.
- `quantity`는 1~5가 고르게 분포한다(1: 143, 2: 154, 3: 143, 4: 167, 5: 157).
- 가장 싼 상품은 5,000원짜리 2개와 10,000원짜리이고 IQR 기준 이상값은 아니다.

### 나의 해석과 판단
이상값은 자동으로 삭제하지 않고 "후보"로 먼저 본다. 0원 상품은 증정품일 수 있고 음수 수량은 반품일 수 있어서, 값만 보고 오류로 단정하면 정상 거래를 지울 수 있기 때문이다. 이번 실습 기준(0 이하 제외)을 적용할 때도 업무 담당자 확인이 먼저 필요하다. 이번 데이터에는 해당 값이 0건이라 실제로 제외된 행은 없었다. 전체 중복과 고유 ID 중복도 구분했다. 전체 행이 같은 중복은 지워도 되지만, ID만 같고 내용이 다른 중복은 자동 삭제하면 정보를 잃는다.

### 업무·분석적 의미
이상값 후보를 따로 목록으로 보관하면 매출 합계가 이상하게 튈 때 어떤 행이 원인인지 바로 추적할 수 있다. 주문 상태를 하나의 표기로 맞추면 `완료`와 `completed`가 따로 집계되어 매출이 쪼개지는 일을 막는다.

### 한계와 추가 확인 사항
IQR이나 0 이하 같은 기준은 통계적·형식적 기준일 뿐 업무적으로 비정상인지는 알려 주지 않는다. 한 주문에서 같은 상품이 여러 번 나오는지, 주문 후 취소된 건의 상세 금액을 어떻게 볼지는 별도 확인이 필요하다.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0건 (`orders.customer_id` → `customers.customer_id`)
- 없는 `order_id`: 0건 (`order_items.order_id` → `orders.order_id`)
- 없는 `product_id`: 0건 (`order_items.product_id` → `products.product_id`)
- 생성한 파생 컬럼: `orders.order_month`(예: 2026-08), `orders.order_dayofweek`(예: Thursday), `order_items.line_total`
- `line_total` 검증: `quantity × unit_price`로 계산. 764행 모두 값이 있고 최솟값 5,000원, 최댓값 1,000,000원, 전체 합계 255,610,000원. 이 합계는 completed·cancelled·refunded 주문이 모두 섞인 값이라 매출이나 순매출이 아님

<!-- 캡처: relationship_checks 표(invalid_count 0,0,0)와 line_total head, 합계 출력 -->
![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
- 세 가지 관계의 `invalid_count`가 모두 0이다.
- `line_total` 결측이 없고 합계는 255,610,000원이다.
- 가격 불일치도 없었다(`order_items.unit_price`와 `products.price`가 764행 모두 같음).
- 상세가 없는 주문도 없다(주문 300건이 모두 `order_items`에 등장).

### 나의 해석과 판단
전처리 중에 행을 지우면(예: 0 이하 상품 제외) 다른 파일이 참조하던 ID가 사라져 관계가 깨질 수 있다. 그래서 전처리가 끝난 뒤 PK/FK를 다시 확인해야 한다. 이번에는 제외된 행이 없어서 0건이 나왔고, 이 결과로 clean 데이터끼리 `merge`해도 행이 사라지거나 늘지 않는다는 것을 확인했다. 또한 `line_total`은 상세 한 행의 수량×단가라는 뜻이고, 합계 255,610,000원을 바로 매출이라고 부르면 취소·환불 주문이 섞이므로 Chapter 06에서는 주문 상태 범위를 정해서 써야 한다.

## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | customers 150 / products 100 / orders 300 / order_items 764 | 동일 (150 / 100 / 300 / 764) | 중복·0 이하 값이 없어 제거된 행이 없음 |
| 결측 | 0 | 0 | 원본에 결측이 없어 대체 규칙이 적용되지 않음 |
| 중복 | 0 | 0 | 전체 행 중복·키 중복 모두 없음 |
| 변환 실패 | 0 | 0 | 숫자·날짜 변환이 모두 성공 |

열 수는 orders가 5 → 7(`order_month`, `order_dayofweek` 추가), order_items가 5 → 6(`line_total` 추가)으로 늘었고 customers(6)와 products(4)는 그대로이다.

<!-- 캡처: comparison 표(rows_raw/rows_processed, columns_raw/columns_processed) -->
![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
- 행 수는 4개 데이터셋 모두 변하지 않았다.
- 열 수만 파생 컬럼 3개만큼 늘었다.
- 결측, 중복, 변환 실패는 처리 전후 모두 0이다.

### 나의 해석과 판단
행 수가 그대로인 것은 의도한 결과다. 이번 데이터에는 삭제할 중복과 기준 이하 값이 없었고, 늘어난 열은 계획한 파생 컬럼 3개와 정확히 일치한다. "행이 줄지 않았다"는 근거는 전체 중복 0건, `price/quantity/unit_price <= 0` 0건, FK 오류 0건이다. 만약 행 수가 줄었다면 어떤 기준으로 몇 건이 빠졌는지 설명할 수 있어야 한다.

### 업무·분석적 의미
전처리 전후가 같다는 것은 이번 샘플 데이터가 원래부터 정제된 상태였다는 뜻이다. 분석 결과(매출 합계 255,610,000원 등)가 전처리 때문에 바뀔 걱정 없이 Chapter 06 EDA를 진행해도 된다.

### 한계와 추가 확인 사항
변화가 0건이라 전처리 코드가 문제를 실제로 잡아내는지는 이 데이터로 검증하지 못했다. 일부러 공백, 쉼표 숫자, `완료` 같은 값을 넣은 테스트 데이터로 확인해 보면 더 확실하다. 또한 `signup_date`가 `order_date`보다 늦은 주문이 56건 있는데(가입 전에 주문한 것으로 보임), 이번 실습 범위의 규칙에는 없는 논리 오류 후보라서 삭제하지 않고 Chapter 06에서 확인할 항목으로 남긴다.

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: `data/processed/customers_clean.csv`, `products_clean.csv`, `orders_clean.csv`, `order_items_clean.csv` (4개)
- 생성된 요약 보고서: `reports/ch05_preprocessing_summary.md`
- `python scripts/preprocess_data.py` 재실행 결과: (직접 실행 후 작성) 오류 없이 종료, 전처리 완료 / 전처리 전후 크기 / 중복 점검 / 관계 점검 / 저장 파일 / 보고서 경로가 출력됨
- 참고(Notebook 21장 실행 결과): `src/preprocessing.py`의 `preprocess_sales_data`로 만든 전후 비교표(150/100/300/764행, 열 6/4/7/6)가 Notebook에서 직접 처리한 결과와 동일하고, 중복 점검도 모두 0이었음
- 재실행 후 달라진 점: 행 수와 합계는 같아야 정상. 달라진 것이 있다면 파일 수정 시각과 요약 보고서의 생성 시각 정도

<!-- 캡처: 터미널에서 python scripts/preprocess_data.py 실행 화면(출력 끝까지, 경로에 개인 폴더명이 보이면 가리기) -->
![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것은 "그때 그 셀 실행 순서와 메모리 상태에서" 성공한 것이다. 셀을 건너뛰거나 순서를 바꾸면 결과가 달라질 수 있고, 다른 사람은 같은 순서를 알 수 없다. 스크립트(`scripts/preprocess_data.py`)는 처음부터 끝까지 같은 순서로 실행되어 같은 clean CSV와 보고서를 만들기 때문에 재현이 된다. 또한 전처리 함수를 `src/preprocessing.py`에 분리하면 같은 코드를 Notebook과 스크립트가 함께 쓰므로 두 결과가 어긋나지 않는다. 저장이 성공했다는 것과 내용이 정상이라는 것도 다르므로 저장한 CSV를 다시 읽어 컬럼과 날짜를 확인하는 단계가 필요하다.

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본(`data/raw`)은 덮어쓰지 않고, 복사본에서 처리한 결과를 `data/processed`에 따로 저장한다.
2. 값을 바꾸거나 지우기 전에 처리 전 상태(크기·결측·중복)를 기록하고, 처리 후에는 변환 실패 건수와 전후 변화를 숫자로 확인한다.
3. 이상값과 중복은 자동으로 지우지 않고 후보로 확인하며, 처리 기준과 이유를 기록해서 스크립트로 다시 실행할 수 있게 한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
`pd.to_numeric` / `pd.to_datetime`의 `errors='coerce'` 이다. 변환하지 못한 값을 오류 없이 조용히 NaN/NaT로 바꿔서, 실패 건수를 세지 않으면 값이 사라진 줄도 모르고 결측 대체나 합계 계산으로 넘어가게 된다. 이번 데이터에서는 실패가 0건이었지만, 실제 데이터였다면 금액이나 날짜가 말없이 빠져 매출과 월별 추이가 틀어졌을 것이다.

### 다음 EDA에서 특히 주의할 데이터 특성
- 전체 `line_total` 합계(255,610,000원)에는 cancelled·refunded 주문이 섞여 있으므로 분석 범위(예: completed만)를 먼저 정해야 한다.
- 10월은 데이터가 2026-10-07까지라 부분 월이므로 월별 비교에 주의한다.
- 가입일이 주문일보다 늦은 주문이 56건 있어 고객 관련 분석 시 확인이 필요하다.
- `order_items.order_id`는 반복되는 것이 정상이므로 주문 건수는 `order_id`의 고유 개수로 센다.

### 현재 전처리의 한계
- 샘플 데이터에 결측·중복·표기 차이가 없어서 결측 대체, 상태값 표준화, 이상값 제외 규칙이 실제로 작동하는 것을 검증하지 못했다.
- 이상값 기준(0 이하, IQR)은 형식적 기준이라 업무적 의미(증정품, 반품 등)는 확인하지 못했다.
- 가입일과 주문일의 선후 관계처럼 파일 사이의 업무 논리는 이번 규칙에서 검증하지 않았다.
- 실제 업무 데이터에서는 결측과 이상값 처리 기준을 담당자와 원본 시스템에 확인해야 한다.

## 최종 제출 체크
- [ ] raw 원본을 덮어쓰지 않았습니다.
- [ ] 처리 전/후 비교를 남겼습니다.
- [ ] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [ ] clean CSV와 재실행 결과를 확인했습니다.
- [ ] 개인정보/Secret이 없습니다.
- [ ] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.